In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (GradientBoostingClassifier as GBR, RandomForestClassifier as RF)

from sklearn.metrics import accuracy_score, classification_report
from ISLP import load_data
from ISLP.models import ModelSpec as MS, Stepwise, sklearn_selected

STATE = 161

In [2]:
# read data table
df = pd.read_table("./magic04.data", sep=',', header=None)

## a. Scale and center df and split into train/test

In [3]:
# scale + center numerical columns
scaler = StandardScaler()

cols = df.select_dtypes(include=['float64']).columns
df[cols] = scaler.fit_transform(df[cols])

In [4]:
# split into train/test
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:-1], df.iloc[:,-1:], test_size=0.2, random_state=STATE)

# Convert y_train and y_test to 1D arrays
flat_y_train = y_train.values.ravel()
flat_y_test = y_test.values.ravel()

# transform response to numeric values
le = LabelEncoder()

y_train_encoded = le.fit_transform(flat_y_train)
y_test_encoded = le.transform(flat_y_test)

## b. Train models on training data & cross validate

In [13]:
# Logistic regression
params = {'C': [0.01, 0.1, 1, 10]}
lr = GridSearchCV(LogisticRegression(max_iter=1000), params, cv=5).fit(X_train, y_train_encoded)

best_lr = lr.best_estimator_

In [15]:
# Linear Discriminant Analysis
lda = LinearDiscriminantAnalysis().fit(X_train, y_train_encoded)

print(f"Accuracy: {lda.score(X_test, y_train_encoded):.4f}")

ValueError: Found input variables with inconsistent numbers of samples: [15216, 3804]

In [7]:
# KNN
params = {'n_neighbors': [3, 5, 11, 21]}
knn = GridSearchCV(KNeighborsClassifier(), params, cv=5).fit(X_train, y_train_encoded)

# Evaluate
print(f"Accuracy: {knn.best_estimator_.score(X_test, flat_y_test):.4f}")

Accuracy: 0.8465


In [8]:
# Linear SVM
params = {'C': [0.1, 1, 10]}
l_svm = GridSearchCV(SVC(kernel='linear', probability=True), params, cv=5).fit(X_train, y_train_encoded)

print(f"Accuracy: {l_svm.best_estimator_.score(X_test, flat_y_test):.4f}")

Accuracy: 0.7950


In [9]:
# Gaussian SVM
params = {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1]}
g_svm = GridSearchCV(SVC(kernel='rbf', probability=True), params, cv=5).fit(X_train, y_train_encoded)

print(f"Accuracy: {g_svm.best_estimator_.score(X_test, flat_y_test):.4f}")

Accuracy: 0.8778


In [10]:
# Random forest
params = {'n_estimators': [50, 100, 200]}
rf = GridSearchCV(RF(), params, cv=5).fit(X_train, y_train_encoded)

best_rf = rf.best_estimator_
print(f"Accuracy: {best_rf.score(X_test, y_test_encoded):.4f}")
print(f"Top Features: {np.argsort(best_rf.feature_importances_)[-3:]}")

Accuracy: 0.8822
Top Features: [1 0 8]


In [11]:
# Gradient Boosting
params = {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1]}
gb = GridSearchCV(GBR(), params, cv=5).fit(X_train, y_train_encoded)

best_gb = gb.best_estimator_
print(f"Accuracy: {best_gb.score(X_test, y_test_encoded):.4f}")
print(f"Top Features: {np.argsort(best_gb.feature_importances_)[-3:]}")

Accuracy: 0.8778
Top Features: [2 0 8]


## c. Applying models to test set

In [12]:
from sklearn.metrics import roc_curve, auc

models = [('LogReg', best_lr), ('LDA', lda), ('KNN', knn.best_estimator_), 
          ('L-SVM', l_svm.best_estimator_), ('G-SVM', g_svm.best_estimator_), 
          ('RF', best_rf), ('GBM', best_gb)]

results = {}
results['AUC'] = []
results['Accuracy'] = []
results['Model'] = []

for name, model in models:
    acc = model.score(X_test, y_test_encoded)
    results['Model'].append(name)
    results['Accuracy'].append(f"{acc:.4f}")

plt.figure(figsize=(8, 6))
for name, model in models:
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_encoded, probs)
    results['AUC'].append(f"{auc(fpr, tpr):.4f}")
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.legend(); plt.show()

TypeError: Labels in y_true and y_pred should be of the same type. Got y_true=[0 1] and y_pred=['g' 'h']. Make sure that the predictions provided by the classifier coincides with the true labels.